<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Lectures/10_Attention_Transformers_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RNN, Attention, Transformers, LLM

Sources:
* Chapter 14, Probabilistic Machine Learning: An Introduction by Kevin Murphy  
* Dive into Deep Learning, by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola https://d2l.ai/index.html
* HuggingFace LLM course https://huggingface.co/learn/llm-course/chapter0/1


Resources:  
[TranformerExplainer_1](https://poloclub.github.io/transformer-explainer/)  
[TranformerExplainer_2](https://simonwillison.net/2024/Aug/11/transformer-explainer/)

Note: In some instances I have directly copied text from sources. This notebook is used for class lecture only.

---
---

# I. Overview of LLM
1. Encoder-Decoder models:
  * Examples: BART, orginal Tranformer (Attenton is ALl you Need paper by Google)
  * It is for seq2seq tasks (following on what we did in RNN), such as translation, summarization, and structured sequence conversion.

2. Decoder only
* Examples: Most AIchatbots: GPT family, LLaMA, Gemma, Mistral
* Open-ended text generation, chat, instruction following, code generation, and reasoning.

3. Encoder only
* Examples: BERT, RoBERTa, DistilBERT
*  For tasks such as text classification, sentiment analysis, named entity recognition, extractive QA, and embeddings/retrieval.


The orginal Transfomers was introduced using Encoder-Decoder mechanisim in the now landmark paper: Attention is All you Need by Vasnawi, et.al., in 2017 (Google). Combines an encoder (using full bidirectional attention over the input) with a decoder (using causal masking and cross-attention to the encoder's output).

This lectures provides an overview of work leading to encoder-decoder transformer, we will look at
* RNN
* RNN with attention
* Self-attention (Attention is all you need! --> Transfomers)

---
---

# II. Data preparation: Text-to-Tensor Pipeline: How LLMs Represent Language
#### NOTE: I am less familiar on 'methods' for tokenization and word embedding, so I used AI help to prepare below. Providing some references for those interested in learning more about these methods
1. **Tokenization** splits raw text into discrete tokens (words or subwords) and maps them to token IDs.
Modern models avoid splitting strictly by whole words (which creates huge vocabularies and out-of-vocabulary issues) or single characters (which makes sequences too long). Instead, they use subword algorithms:
* Byte-Level BPE (Byte-Pair Encoding): Starts with individual bytes (256 base tokens) and iteratively merges the most frequent adjacent pairs. Used in GPT models, Llama, and Qwen; guarantees zero unknown (<unk>) tokens since any text can be represented as bytes.
* Unigram Language Model / SentencePiece: Starts with a large seed vocabulary and iteratively prunes tokens that least impact overall corpus likelihood. Often paired with SentencePiece to treat whitespace as a regular character, avoiding language-specific pre-tokenization. Used in Gemma, T5, and Mistral.
* WordPiece: Similar to BPE, but merges pairs based on maximizing likelihood rather than raw frequency. Used in BERT-family models.

2. **Token embeddings** Once text becomes a sequence of integer token IDs, embeddings map them into continuous vector space:
* Learned Lookup Tables: A trainable matrix of size \(V \times d_{\text{model}}\) (vocabulary size × hidden dimension). Each token ID retrieves its corresponding row vector, which is updated via backpropagation during pre-training.
* Weight Tying: Many architectures share the exact same weight matrix between the input token embedding layer and the final output projection layer (LM head) to save parameters and stabilize training.
* Sub-token & Multimodal Projections: For non-text inputs (images, audio) or byte-level/patch-level models, lightweight linear layers or small encoders project raw patches directly into the shared $d_{\text{model}}$ embedding space.

3. **Positional encoding**
The performance of “vanilla” self-attention can be low, since attention is permutation invariant, and hence ignores the input word ordering. In "The dog chased another dog" outputs from multi-head attention will be identical for both 'dog'
To overcome this, we can concatenate (old methdod) or add (newer methods) the word embeddings with a positional embedding, so that the model knows what order the words occur in. This is done by injecting position information (via fixed sine/cosine functions, learned position embeddings, or relative/rotary methods like RoPE).
Early Transformers added fixed sinusoidal or learned absolute position vectors directly to token embeddings before the first layer. Modern LLMs almost exclusively inject relative position inside every self-attention layer instead, allowing models to generalize to longer sequences.

[SEE HUGGING FACE VIZUAL](https://huggingface.co/blog/designing-positional-encoding)

* References:
  * For #1 Reference: Radford et.al., Language Models are Unsupervised Multitask Learners https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
  * #2 and 3, below would be a good read to understand what was used and modifications made to #3.
  Vasvani, et. al., Attention is All you Need, https://arxiv.org/pdf/1706.03762

---
---

# III. What is the **Seq2Seq problem** or task of **Encoder-Decoder Models**:
Learn functions of the form $f(θ) : \mathbb{R}^{TD} → \mathbb{R}^{T'C}$. i.e., map from one sequence of length $T$ to another of length $T ′$

**Seq2Seq problem** typically rely on **encoder-decoder mechanism**:
* *Encoder: The encoder receives an input and builds a representation of it (its features). This means that the model is optimized to acquire understanding from the input.*
* *Decoder: The decoder uses the encoder’s representation (features) along with other inputs to generate a target sequence. This means that the model is optimized for generating outputs.*

---

**Common architectures of encoder-decoder mechanism**
* CNN with transposed convolution (U-Net) (we saw this lecture on CNN/ DL for image data)
* RNN as encoder and decoder
* RNN with attention
* Transformers (multi-head self-attention)(Attention is all you need, by Vaswani et. al., 2017)(Google)

---
---



# IV. Seq2Seq RNN as encoder and decoder**

  * **Aligned case (we saw this in RNN lecture)**: $ T ′ = T $, i.e., input and output sequences have the same length
  * **Unaligned case**: $T' \not= ̸T $, i.e., input and output sequences have different lengths. This is the scenario for Natural Language Processing

**Unaligned case (most seq2seq Natural Language Processing tasks fall under this)**  
* **Encoder**: Suppose the input sequence is $x_1,...x_T$, such that $x_t$ is the $i^{th}$ token. At time step $t$, the RNN transforms the input feature vector $\mathbf{x_t}$ for $x_t$ and the hidden state $h_{t-1}$ from the previous time step into the current hidden state $h_t$ (this is similar to what we saw in RNN lectures).
we have $$h_t=f(\mathbf{x_t},h_{t-1})$$

* **Context variable**: What is new here: the encoder transforms the hidden states at all time steps into a **context variable** through a customized function
$$c=q(h_1,...,h_T)$$
The context variable could be just the hidden state corresponding to the encoder RNN’s representation after processing the final token of the input sequence (last state of an RNN or average pooling over a biRNN).

* **Decoder**: The output sequence is generate using an RNN as a decoder
$y_{1:T ′} = f_d(c)$


<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.7.png?raw=true" height="200" width ="500">

*Embedded from TextBook; Figure 15.7: Encoder-decoder RNN architecture for mapping sequence $x_{1:T}$ to sequence $y_{1:T ′}$*.

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.8_A.png?raw=true" height="400" width ="500">

*Figure 15.8: llustration of a seq2seq model for translating English to French. The - character represents the end of a sentence.*

---
---

# V. **Drawbacks of Seq2Seq RNN**:

**Drawback**: seq2seq model above used an RNN decoder of the form $h^d_t = f_d(h^d_{t−1}, y_{t−1}, c)$, where $c$ is a fixed-length context vector, representing the encoding of the input
$x_{1:T}$ . Usually we set $c = h^e_T$, which is the final state of the encoder RNN (or we use a bidirectional RNN with average pooling).

However, for tasks such as machine translation, this can result in poor performance, since the output does not have access to the input words themselves.

---
---


# VI. RNN with Attention

We can avoid the bottleneck of seq2seq RNN  *by allowing the output words to directly “look at” the input words. But which inputs should it look at? Word order is not always preserved across languages (e.g., German often puts verbs at the end of a sentence), so we need to infer the alignment between source and target.*

This problem is solved (in a differentiable way) by using **(soft) attention**.

In particular, we can **replace the fixed context vector $c$ in the decoder with a dynamic context vector $c_t$ computed using attention**

# What is attention? We will first look at this queston before going to RNN with Attention

---

# VII. **ATTENTION**

Attention was first introduced in 2014 and 2015
* Bahdanau et. al., Neural Machine Translation by Jointly Learning to Align and Translate, 2014 https://arxiv.org/pdf/1409.0473
* Luong et.al., Effective Approaches to Attention-based Neural Machine Translation, https://arxiv.org/abs/1508.04025   

$\color{orange}{\text{Attention is an operator similar to how convolution was an operator was used in CNN.}}$

**Conceptual Overview of Attention**:
In the neural networks thus far, we calculated hidden activations as a linear combination of the previous layer ($X$) with weights ($W$) passed through a nonlinear activation function: $Z = φ(XW)$

What attention does is to formulate an operator where the weights depend on the inputs, i.e., get a multiplicative form $Z = φ(XW(X))$. This is typically done as $Z = φ(VW(Q, K))$, where
* $Q$ are a set of queries used to describe what each input is “looking for” (Figure below depicts each token (x), in matrix form we take the full (X),
* $K$ are a set of keys used to describe what each input vector contains, and
* $V$ are a set of values  used to describe how each input should be transmitted to the output.

We will first look at the two types of attention that were orginally proposed and then look at the scaled dot-product which is the one most commonly used now.

---

1. **Bahdanau (additive) attention** https://arxiv.org/pdf/1409.0473

**General equation**:
$$Attn(q, (k_1, v_1), . . . , (k_m, v_m)) = Attn(q, (k_{1:m}, v_{1:m})) =
\sum_i^m \alpha_i(q, k_{1:m})v_i $$
where
* $\alpha_i(q, k_{1:m})$ is the $i$’th **attention weight**
*  $0 ≤ \alpha_i(q, k_{1:m}) ≤ 1$ for each $i$
* $\sum_i \alpha_i(q, k_{1:m}) = 1$

The **attention weights** are defined as:
$$
\alpha_i(q, k_{1:m})
= \text{softmax}_i\big( [\, a(q, k_1), \ldots, a(q, k_m) \,] \big)
= \frac{\exp(a(q, k_i))}{\sum_{j=1}^m \exp(a(q, k_j))}.
$$
where,
* $a(q, k)$ are the **attention scores**  

**Attention scores** compute the similarity between query $q$ and key $k$
* **Bahdanau (additive) attention**: Computes attention score by using a feed-forward network with
a single hidden layer.
  $$a(q,k)=w_v^T tanh(W_qq+W_kk)$$
* $W_q$ and $W_k$ are learnable weight matrices
* In this general case, the query $q ∈ R^q$ and the key $k ∈ R^k$ may
have different sizes. Then $W_q\in \mathbb{R}^{h\times q}$; $W_k\in \mathbb{R}^{h\times k}$  

---
2. **Luong attention (general)**  
${\displaystyle {\text{Attention}}(Q,K,V)={\text{softmax}}(QWK^{T})V}$
* $W$ is learnable weight matrix

---
3. **Scaled dot-product attention**  
Introduced in context of [Attention is all you need by Vaswani et. al.](https://arxiv.org/pdf/1706.03762). Similar to Luong attention, except queries and keys both have length $d$ and the score is scaled by ${√d}$.

$$Attn(Q,K,V) = softmax\big(\frac{QK^T}{√d} \big) V $$
* $a(q,k)=\big(\frac{QK^T}{√d} \big)$ are the attention scores, it computes the similarity between query $q$ and key $k$;
*  $\alpha(q,k)=softmax\big(\frac{QK^T}{√d} \big)$ are the attention weights
* Further, they  can be self-attention or cross-attention
  
  * **Self-attention**: if query, key, and values all come from $X$, i.e., they are linear projections of the input (note: inputs can be outputs from previous layers). (See later section on transfomer: Self-attention is used in encoder, and masked self-attention is used in decoder).
    * $Q = W_qX$,
    * $K = W_kX$, and
    * $V = W_vX$.
  * **Cross-attention**: See later section on transfomer: cross-attention is used in encoder-decoder merge: $Q$ comes from decoder, $K$ and $V$ come from encoder.
---
---

**VISUAL UNDERSTANDING OF ATTENTION**
<img src="https://upload.wikimedia.org/wikipedia/commons/8/81/Attention-qkv.png?raw=true" height="400" width ="800">  
Source: Embedded from Wikipedia: https://en.wikipedia.org/wiki/Attention_(machine_learning)  



[**EXAMPLE CODE**](https://colab.research.google.com/github/probml/pyprobml/blob/master/notebooks/book1/15/attention_torch.ipynb)



**Padded masking**
Sequences are typically of differnt lengths. We might want to pad sequences to a fixed length (for efficient minibatching), in which case we should “mask out” the padded locations. This is called masked attention. We can immplement this efficiently by setting the attention score for the masked entries to a large negative number, such as $−10^6$, so that the corresponding softmax weights will be 0.

**Masked Attention**: Note: above masking mechanism is also used for **Casually Masked Attention** in **decoder LLMs**. All-to-all attention cannot be done during the autoregressive decoding process, i.e., the decoder cannot attend to future outputs that have yet to be decoded. Thus, mask matrix ($M$) is added to attention. $$Attn(Q,K,V) = softmax\big(\frac{QK^T}{√d} + M \big) V $$

---
---




# VIII. **Seq2Seq: RNN with attention** (original versions used the Bahdanau or Luong attention)

Recall in **seq2seq RNN** model above : It used an RNN decoder of the form $h^d_t = f_d(h^d_{t−1}, y_{t−1}, c)$, where $c$ is a fixed-length context vector, representing the encoding of the input
$x_{1:T}$ . Usually we set $c = h^e_T$, which is the final state of the encoder RNN (or we use a bidirectional RNN with average pooling).

In **RNN with attention**, we can **replace the fixed context vector $c$ in the decoder with a dynamic context vector $c_t$ computed using attention** as follows:
$$c_t = \sum_{i=1}^T \alpha_i(h^d_{t−1}, h^e_{1:T})h^e_i$$
where,
* the **query** is the hidden state of the decoder at the previous step, $h^d_{t−1}$,
* the **keys** are all the hidden states from the encoder ($h^e_{1:T}$), and
* the **values** are also the hidden states from the encoder.
(When the RNN has multiple hidden layers, we usually take the top layer from the encoder, as the keys and values, and the top layer of the decoder as the query.)
* This **context vector** is concatenated with the input vector of the decoder, $y_{t−1}$, and fed into the decoder, along with the previous hidden state $h^d_{t−1}$, to create $h^d_t$ .

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.18.png?raw=true" height="400" width ="600">

*Embedded from TextBook; Figure 15.18: Illustration of seq2seq with attention for English to French translation*

We can train this model in the usual way on sentence pairs, and then use it to perform machine
translation. (See [RNN with attention-based decoder EXAMPLE CODE](https://colab.research.google.com/github/probml/pyprobml/blob/master/notebooks/book1/15/nmt_attention_torch.ipynb). We can also visualize the attention weights computed at each step of decoding, to get an idea of which parts of the input the model thinks are most relevant for generating the corresponding output. Some examples are shown in Figure 15.19

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.19_A.png?raw=true" height="400" width ="500">

*Embedded from TextBook; Figure 15.18: Illustration of the attention heatmaps (attention weights ~softmax of attention scores) generated while translating a sentence from Spanish to English.*

* [EXAMPLE Code](https://www.tensorflow.org/text/tutorials/nmt_with_attention)

---
---
**Soft v hard attention**  
If we force the attention heatmap to be sparse, so that each output can only attend to one input location instead of a weighted combination of all of them, the method is called hard attention. Below figure compares these two approaches for an image captioning problem. Unfortunately, hard
attention results in a nondifferentiable training objective, and requires methods such as reinforcement learning to fit the model.
<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.22_A.png?raw=true" height="400" width ="500">

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.22_B.png?raw=true" height="400" width ="500">

*Embedded from TextBook; Figure 15.22: Image captioning using attention. (a) Soft attention. Generates “a woman is throwing a frisbee in a park”. (b) Hard attention. Generates “a man and a woman playing frisbee in a field”*

---
---

# IX. **Seq2vec with attention (text classification)**
We can also use attention with sequence classifiers. One study applied an RNN classifier
to the problem of predicting if a patient will die or not. The input is a set of electronic health
records, which is a time series containing structured data, as well as unstructured text (clinical
notes). Attention is useful for identifying “relevant” parts of the input.

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.20.png?raw=true" height="500" width ="900">

*Embedded from TextBook; Figure 15.20: Example of an electronic health record. In this example, 24h after admission to the hospital, the RNN classifier predicts the risk of death as 19.9%; the patient ultimately died 10 days after admission. The “relevant” keywords from the input clinical notes are shown in red, as identified by an attention mechanism.*

---
---


# X. **TRANSFORMER**

**<font color="#009688">The transformer model is a seq2seq model which uses attention in the encoder as well as the decoder, thus eliminating the need for RNNs.</font>**

We will first look at a few concepts before putting the concepts together to for the Transformer

## Xa. Attention v self-attention
<font color="#009688">Attention (used in RNN with attention):</font> : In the context of seq2seq using RNN with attention, the decoder of an RNN used attention to the input sequence in order to capture contextual embeddings of each input. (In modern day -after 'Attention is All you Need' paper- this would be called cross-attention, but back then it was just called attention.)

**<font color="#009688"> Self-attention </font>** (see earlier section where self-attention was defined)  
Rather than only having the decoder attend to the encoder, we can remove the RNN recurrence and allow the encoder to attend to itself—and the decoder to attend to its own previously generated tokens—to capture context directly. This is called self-attention (see earlier discussions in ATTENTION overview).

<font color="#009688">Self-attention in encoder:</font>
Given a sequence of input tokens $x_1, . . . , x_n$, where $x_i ∈ \mathbf{R}^d$, self-attention can generate a sequence of outputs of the same size using
$y_i = Attn(x_i, (x_1, x_1), . . . , (x_n, x_n))$
where, the query $Q$ is $x_i$, and the keys $K$ and values $V$ are all the (valid) inputs $x_1, . . . , x_n$.

<font color="#009688"> Masked Self-attention in decoder:</font>: All-to-all attention cannot be done during the autoregressive decoding process, i.e., the decoder cannot attend to future outputs that have yet to be decoded. To use self-attention in a decoder, we can set $x_i = y_{i−1}$, and $n = i − 1$, so all the previously generated outputs are available. This is done by adding a mask matrix ($M$) to attention, where $M_{0:i-1,i}=0$ and $M_{i:,i}=\infty$. $$Attn(Q,K,V) = softmax\big(\frac{QK^T}{√d} + M \big) V $$ At training time, all the outputs are already known, so we can evaluate the above function in parallel, overcoming the sequential bottleneck of using RNNs.

<font color="#009688">Encoder-Decoder Cross-Attention.</font> The decoder provides the Queries $(Q$), while the encoder's output provides the Keys $(K)$ and Values ($V$). This lets the decoder focus on the most relevant parts of the input sequence as it generates each output word.


**Advantages of self-attention**
* Naturally allows for parallel compute during training.
* Improved representations of context:  As an example, consider translating the English sentences “The animal didn’t cross the street because
it was too tired” and “The animal didn’t cross the street because it was too wide” into French. To generate a pronoun of the correct gender in French, we need to know what “it” refers to (this is called
**coreference resolution**). In the first case, the word “it” refers to the animal. In the second case, the word “it” now refers to the street.

Figure 15.23 illustrates how self attention applied to the English sentence is able to resolve this ambiguity. In the first sentence, the representation for “it” depends on the earlier representations of
“animal”, whereas in the latter, it depends on the earlier representations of “street”.

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.23.png?raw=true" height="200" width ="500">

*Embedded from TextBook; Figure 15.23:llustration of how encoder self-attention for the word “it” differs depending on the input context.*

---



## Xb. **Multi-head attention**  
Basic idea behind multi-headed attention (MHA).
* We can think of an attention matrix like a kernel matrix;
  * In kernel methods (e.g., RBF or polynomial kernels), a kernel function $K(x_i, x_j)$ measures the similarity between two data points $x_{i}$and $x_{j}$ in a high-dimensional space.  
* Then, using multiple attention matrices can capture different notions of similarity.
  * This is analogous to how we used different filters in CNN to generate different feature maps or channels.
  * Example: In machine tranlsation, each head can focus on a different linguistic relationship—such as word meaning, grammar, gender agreement, or word order—at the exact same time.
    * Suppose translating English: "The black cat sat on the warm mat because it was tired."  $\rightarrow$  French: "Le chat noir s'est assis sur le tapis chaud parce qu'il était fatigué."
    * **Head 1: Coreference** — Finds what a pronoun refers to by connecting `"it"` $\leftrightarrow$ `"cat"`, which determines whether `"it"` translates to masculine **`"il"`** (referring to *le chat*) or feminine **`"elle"`**.
    * **Head 2: Grammar & Agreement** — Links adjectives to their nouns by connecting `"black"` $\leftrightarrow$ `"cat"`, ensuring `"black"` agrees in gender and number with `"cat"` (**`"noir"`**, masculine singular, rather than `"noire"`).
    * **Head 3: Action & Subject** — Connects verbs to who performed them by linking `"sat"` $\leftrightarrow$ `"cat"`, conjugating the verb properly for a singular subject (**`"s'est assis"`**).
    * **Head 4: Local Word Order** — Looks at immediate neighbors by connecting `"the"` $\leftrightarrow$ `"black"` $\leftrightarrow$ `"cat"`, handling word-order swaps between languages (English `"black cat"` $\rightarrow$ French `"chat noir"`).


**MHA**: Given queries $q \in \mathbb{R}^{d_q}$, keys $k_j \in \mathbb{R}^{d_k}$, and values $v_j \in \mathbb{R}^{d_v}$, we define the $i$-th attention head as:

$$
h_i = \text{Attn}\big(W_i^{(q)} q,\; \{ W_i^{(k)} k_j,\; W_i^{(v)} v_j \}\big) \in \mathbb{R}^{p_v}
$$

where  
$W_i^{(q)} \in \mathbb{R}^{p_q \times d_q}$,  
$W_i^{(k)} \in \mathbb{R}^{p_k \times d_k}$,  
and $W_i^{(v)} \in \mathbb{R}^{p_v \times d_v}$  
are projection matrices.

We then stack the $h$ heads together and project to $\mathbb{R}^{p_o}$ using:

$$
h = \text{MHA}(q,\{k_j, v_j\}) = W_o
\begin{pmatrix}
h_1 \\
\vdots \\
h_h
\end{pmatrix}
\in \mathbb{R}^{p_o}
$$

where each $h_i$ is defined above, and $W_o \in \mathbb{R}^{p_o \times h p_v}$.

If we set $p_qh = p_kh = p_v h= p_o$,  we can compute all the output heads in parallel.

[**CODE EXAMPLE**](https://colab.research.google.com/github/probml/pyprobml/blob/master/notebooks/book1/15/multi_head_attention_torch.ipynb)

---


##  **Putting it all together --> Transformer**

A transformer is a seq2seq model that uses self-attention for the encoder and decoder rather than an RNN. The encoder uses a series of encoder blocks, each of which uses multi-headed attention, residual connections, feedforward layers, and layer normalization
```
def EncoderBlock(X):
  Z = LayerNorm(MultiHeadAttn(Q=X, K=X, V=X) + X)
  E = LayerNorm(FeedForward(Z) + Z)
return E

def Encoder(X, N):
  E = POS(Embed(X))
for n in range(N):
  E = EncoderBlock(E)
return E

def DecoderBlock(Y, E):
  Z = LayerNorm(MultiHeadAttn(Q=Y, K=Y, V=Y) + Y)
  Z’ = LayerNorm(MultiHeadAttn(Q=Z, K=E, V=E) + Z)
  D = LayerNorm(FeedForward(Z’) + Z’)
return D

def Decoder(Y, E, N):
  D = POS(Embed(Y))
for n in range(N):
  D = DecoderBlock(D,E)
return D
```

---
---

## TRANSFORMER for Encoder-Decoder model

<img src="https://github.com/diseasemodeling/MIE_525_625/blob/main/figs/Multi-head-attention.png?raw=true" height="500" width ="800">

*Embedded from TextBook; Figure 15.26: The Transformer*


1. The Left Stack: Encoder
* Inputs & Embeddings. The source sequence (such as a sentence to be translated) enters at the bottom, is converted into numerical vectors via an embedding layer, and receives Positional Encoding so the model knows the order of the tokens.
* <font color="#009688">Self-Attention & Feed-Forward.</font> Inside the repeated encoder blocks (Nx), Multi-Head Attention lets every input token look at all other input tokens simultaneously to build context (no mask), followed by a position-wise feed-forward network.

* Context Output. The final output of the encoder stack is passed across to the middle of the decoder stack.

2. The Right Stack: Decoder
* Outputs (Shifted Right). During training, the target output sequence (the sentence in the langugae the input should be translated to) is fed into the bottom of the decoder shifted one position to the right (prepended with a start-of-sequence token). This ensures that when the model predicts the token at position $i$, it only has access to the ground-truth tokens from positions $0$ through $i-1$.
* <font color="#009688">Masked Multi-Head Attention.</font> The first sub-layer in the decoder applies a mask to future positions so the decoder cannot "cheat" by looking ahead at words it hasn't generated yet.
* <font color="#009688">Encoder-Decoder Cross-Attention.</font> The middle sub-layer combines both stacks: the decoder provides the Queries $(Q$), while the encoder's output provides the Keys $(K)$ and Values ($V$). This lets the decoder focus on the most relevant parts of the input sequence as it generates each output word.
* Linear & Softmax. At the top of the decoder, a linear layer and softmax function convert the final vectors into probabilities to predict the next token

<font color="#009688">Output probabilities</font>: similar to a multiclass classification problem, it corresponds to teh probability of belonging to a class. Here, we dont use the term 'class' we instead use the term <font color="#009688">'token'</font>. So the size of the probability vector (size of output layer) is the number of tokens in the <font color="#009688">**vocabulary**</font> used by the model.

<font color="#009688">**Vocabulary**</font>: Modern LLMs use subword vocabularies—typically ranging from 32,000 to over 250,000 tokens—built from individual characters, common word pieces, whole frequent words, punctuation, and special control tokens. Rather than storing every whole word in a dictionary (which would require millions of entries and fail on typos or new words) or using single characters (which makes sequences too long), LLMs split text into statistically common chunks.

---
---

# XI. **Comparing transformers, CNNs and RNNs**
The figure compares three different architectures for mapping a sequence $x_{1:n}$ to another sequence $y_{1:n}$: a 1d CNN, an RNN, and an attention-based model.
<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_15.27.png?raw=true" height="300" width ="500">
Source: Embedded from textbook *Figure 15.27: Comparison of (1d) CNNs, RNNs and self-attention models.*

---
---

# X. **LARGE LANGUAGE MODELS (LLMs)**

**Transformers (Attention is all you need, by Vaswani et. al., 2017)**
https://arxiv.org/pdf/1706.03762  
The article that started it all. Original article mainly motivated for machine translation (encoder-decoder mechanism) also proposed use of tranformers in encoder-only and decoder-only mechanisms and **proposed** the future that we know today "*We are excited about the future of attention-based models and plan to apply them to other tasks. We plan to extend the Transformer to problems involving input and output modalities other than text and to investigate local, restricted attention mechanisms to efficiently handle large inputs and outputs
such as images, audio and video. Making generation less sequential is another research goals of ours.*"

---

**Large Language Models (LLM) (models built using transformer architectures) early history**
* (Attention is all you need, by Vaswani et. al., 2017)(Google)
* *June 2018: GPT, the first pretrained Transformer model, used for fine-tuning on various NLP tasks and obtained state-of-the-art results* **they are generative LLM or casual LLM or autoregresive model built on decoder-only transformer**
* *October 2018: BERT, another large pretrained model, this one designed to produce better summaries of sentences* (**non-generative LLMs for learning representations; build on encoder-only transformer**)
* *February 2019: GPT-2, an improved (and bigger) version of GPT that was not immediately publicly released due to ethical concerns*
* *October 2019: T5, A multi-task focused implementation of the sequence-to-sequence Transformer architecture.*
* 2019: BART, or Bidirectional and Auto-Regressive Transformers. It is a denoising autoencoder that combines the strengths of BERT's bidirectional encoder for understanding text and GPT's auto-regressive decoder for generating text.
* *May 2020, GPT-3, an even bigger version of GPT-2 that is able to perform well on a variety of tasks without the need for fine-tuning (called zero-shot learning)*
* *January 2022: InstructGPT,  Use reinforcement learning with human feedback (RLHF) for fine-tuning GPT 3 to follow an instruction in a prompt and provide a detailed response.*
* *November 2022: **ChatGPT*** Instruction fine-tuning (type of finetuning that teaches model to behave like chatbot) turned GPT 3 into ChatGPT
<img src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter1/transformers_chrono-dark.svg?raw=true" height="400" width ="800">  
Source: Embedded from [HuggingFace LLM Course](https://huggingface.co/learn/llm-course/chapter1/3)

---
---

# [TranformerExplainer](https://simonwillison.net/2024/Aug/11/transformer-explainer/)


# XII. Overview of LLM

What we discussed above was the original Transformer introduced in the 'Attention is All you Need' paper. It was a encoder-decoder model.

## Common architectures
1. Encoder-Decoder models :
  * Examples: BART, orginal Tranformer (Attenton is ALl you Need paper by Google)
  * It is for seq2seq tasks (following on what we did in RNN), such as translation, summarization, and structured sequence conversion.

2. Decoder only
* Examples: Most AIchatbots: GPT family, LLaMA, Gemma, Mistral
* Open-ended text generation, chat, instruction following, code generation, and reasoning.

3. Encoder only
* Examples: BERT, RoBERTa, DistilBERT
*  For tasks such as text classification, sentiment analysis, named entity recognition, extractive QA, and embeddings/retrieval.

See example architectures from HuggingFace for each of above.

In [1]:
from transformers import AutoConfig, AutoModel, AutoModelForCausalLM, AutoModelForSeq2SeqLM

# 1. Encoder-Only: BERT (bert-base-uncased)
enc_config = AutoConfig.from_pretrained("bert-base-uncased")
enc_model = AutoModel.from_config(enc_config)

# 2. Decoder-Only: GPT-2 (gpt2)
dec_config = AutoConfig.from_pretrained("gpt2")
dec_model = AutoModelForCausalLM.from_config(dec_config)

# 3. Encoder-Decoder: T5 (t5-small)
enc_dec_config = AutoConfig.from_pretrained("t5-small")
enc_dec_model = AutoModelForSeq2SeqLM.from_config(enc_dec_config)

# Print key hyperparameters (N layers, d_model, h heads)
for name, cfg in [("Encoder-Only (BERT)", enc_config),
                  ("Decoder-Only (GPT-2)", dec_config),
                  ("Encoder-Decoder (T5)", enc_dec_config)]:
    print(f"=== {name} ===")
    print(f"Layers (N): {getattr(cfg, 'num_hidden_layers', getattr(cfg, 'n_layer', None))}")
    print(f"Hidden Size (d_model): {getattr(cfg, 'hidden_size', getattr(cfg, 'n_embd', None))}")
    print(f"Attention Heads (h): {getattr(cfg, 'num_attention_heads', getattr(cfg, 'n_head', None))}\n")

# Print the full PyTorch module tree for each
print("--- BERT Architecture ---")
print(enc_model)

print("\n--- GPT-2 Architecture ---")
print(dec_model)

print("\n--- T5 Architecture ---")
print(enc_dec_model)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

=== Encoder-Only (BERT) ===
Layers (N): 12
Hidden Size (d_model): 768
Attention Heads (h): 12

=== Decoder-Only (GPT-2) ===
Layers (N): 12
Hidden Size (d_model): 768
Attention Heads (h): 12

=== Encoder-Decoder (T5) ===
Layers (N): 6
Hidden Size (d_model): 512
Attention Heads (h): 8

--- BERT Architecture ---
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768

## Loss: cross-entorpy loss and perplexity

At each position the model produces logits $z \in \mathbb{R}^{V}$, one score for every token in the vocabulary of size $V$. A softmax turns these into probabilities:

$$p_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}$$

The loss at that position is the negative log probability of the correct token $y$:

$$\mathcal{L} = -\log p_y$$

This is averaged over every position that counts toward the loss (and over the batch):

$$\mathcal{L} = -\frac{1}{N}\sum_{t=1}^{N} \log p_\theta(y_t \mid \text{context}_t)$$

Evaluation metric typically used for language models:
<font color="#009688">**perplexity**
 $=e^{\mathcal{L}}$ </font>


### Specific to architecture

- **Decoder-only (GPT-style, causal LM):** next-token prediction. Position $t$ predicts token $t+1$, so the targets are the inputs shifted left by one. A causal mask stops each position from seeing future tokens. Every position contributes to the loss.
- **Encoder-only (BERT-style, masked LM):** about 15% of the tokens are masked, and the loss is computed only at those masked positions. The original BERT also had a next-sentence-prediction loss, but later models such as RoBERTa dropped it.
- **Encoder–decoder (original Transformer, T5):** the decoder predicts the target sequence one token at a time, given the encoded source. It is trained with *teacher forcing*, meaning the decoder is fed the true previous target tokens rather than its own predictions.